In [ ]:
import numpy as np
import os
import json
import shutil

In [ ]:
# CHECKING THE INPUT BDD100K ANNOTATIONS

from pprint import pprint

json_path = "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json"

with open(json_path, "r") as f:
    annotations = json.load(f)

print("Total images:", len(annotations))
print()

print("Keys in first image:")
print(annotations[0].keys())

print("\nFirst annotation:\n")
pprint(annotations[0])

In [ ]:
# FINDING ALL DETECTION CLASSES

classes = set()

for image in annotations:

    for label in image["labels"]:

        if "box2d" in label:
            classes.add(label["category"])

print("Detection Classes:\n")

for c in sorted(classes):
    print(c)

In [ ]:
# CHECKING IMAGE DIMENSIONS

img = cv2.imread(
    "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/002b485a-3f6603f2.jpg"
)

print("Shape:", img.shape)

In [ ]:
# DEFINING THE DATASET PATHS

ROOT = "/kaggle/input/datasets/solesensei/solesensei_bdd100k"

TRAIN_JSON = os.path.join(
    ROOT,
    "bdd100k_labels_release",
    "bdd100k",
    "labels",
    "bdd100k_labels_images_train.json"
)

VAL_JSON = os.path.join(
    ROOT,
    "bdd100k_labels_release",
    "bdd100k",
    "labels",
    "bdd100k_labels_images_val.json"
)

TRAIN_IMAGES = os.path.join(
    ROOT,
    "bdd100k",
    "bdd100k",
    "images",
    "100k",
    "train"
)

VAL_IMAGES = os.path.join(
    ROOT,
    "bdd100k",
    "bdd100k",
    "images",
    "100k",
    "val"
)

OUTPUT_DIR = "/kaggle/working/BDD100K_YOLO"

In [ ]:
# DEFINING RESTRICTED 10 CLASSES ON WHICH WE WILL TRAIN OUR MODEL

CLASSES = [
    "bike",
    "bus",
    "car",
    "motor",
    "person",
    "rider",
    "traffic light",
    "traffic sign",
    "train",
    "truck"
]

# GIVING ABOVE CLASSES A UNIQUE ID RANGING FROM 0 TO 9 SO THAT YOLO CAN UNDERSTAND THEM EASILY
CLASS2ID = {
    cls: idx
    for idx, cls in enumerate(CLASSES)
}

print(CLASS2ID)

In [ ]:
# JUST INSPECTING THE NUMBER OF ANNOTATIONS

with open(TRAIN_JSON, "r") as f:
    train_annotations = json.load(f)

with open(VAL_JSON, "r") as f:
    val_annotations = json.load(f)

print(len(train_annotations))
print(len(val_annotations))

In [ ]:
# CREATING TRAIN AND VAL FOLDERS FOR IMAGES,LABELS (THESE WILL HAVE .txt FILES HAVING THE NECESSARY INPUT FOR YOLO)

os.makedirs(f"{OUTPUT_DIR}/images/train", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/images/val", exist_ok=True)

os.makedirs(f"{OUTPUT_DIR}/labels/train", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels/val", exist_ok=True)

In [ ]:
# CREATIN A LOOKUP DICTIONARY FOR QUICK CHECK OF trainA, trainB, testA, testB FOLDER IMAGES 

from pathlib import Path

train_lookup = {}

for folder in ["trainA", "trainB"]:
    
    folder_path = Path(TRAIN_IMAGES) / folder
    
    for img in folder_path.glob("*.jpg"):
        
        train_lookup[img.name] = str(img)

print(f"Training images indexed: {len(train_lookup)}")

In [ ]:
val_lookup = {}

for img in Path(VAL_IMAGES).glob("*.jpg"):
    
    val_lookup[img.name] = str(img)

print(f"Validation images indexed: {len(val_lookup)}")

In [ ]:
# DEFINING A METHOD WHICH WILL FILL THE IMAGES,LABELS FOLDERS WITH .txt FILES IN YOLO INPUT FRIENDLY FORMAT, 
# SUCH AS CREATING THE X CENTRE, Y CENTRE AS YOLO DOESN'T ACCEPT THE CORNERED POINTS DIRECTLY AS x1, y1, x2, y2

from tqdm import tqdm

def convert_to_yolo(annotations, image_lookup, split):

    images_out = os.path.join(OUTPUT_DIR, "images", split)
    labels_out = os.path.join(OUTPUT_DIR, "labels", split)

    count = 0

    for ann in tqdm(annotations):

        filename = ann["name"]

        # Skip missing images
        if filename not in image_lookup:
            continue

        image_path = image_lookup[filename]

        label_lines = []

        for obj in ann["labels"]:

            if "box2d" not in obj:
                continue

            cls = obj["category"]

            if cls not in CLASS2ID:
                continue

            box = obj["box2d"]

            x1 = box["x1"]
            y1 = box["y1"]
            x2 = box["x2"]
            y2 = box["y2"]

            # Image size (BDD100K)
            W = 1280
            H = 720

            x_center = ((x1 + x2) / 2) / W
            y_center = ((y1 + y2) / 2) / H
            width = (x2 - x1) / W
            height = (y2 - y1) / H

            class_id = CLASS2ID[cls]

            label_lines.append(
                f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
            )

        # Skip images without detection objects
        if len(label_lines) == 0:
            continue

        # Copy image
        shutil.copy(
            image_path,
            os.path.join(images_out, filename)
        )

        # Save label
        txt_name = filename.replace(".jpg", ".txt")

        with open(os.path.join(labels_out, txt_name), "w") as f:
            f.write("\n".join(label_lines))

        count += 1

    print(f"\nFinished {split}: {count} images")

In [ ]:
convert_to_yolo(
    train_annotations,
    train_lookup,
    "train"
)

In [ ]:
convert_to_yolo(
    val_annotations,
    val_lookup,
    "val"
)

In [ ]:
# CREATING THE dataset.yaml NEEDED FOR THE YOLO

import yaml

dataset_yaml = {
    "path": OUTPUT_DIR,
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "bike",
        1: "bus",
        2: "car",
        3: "motor",
        4: "person",
        5: "rider",
        6: "traffic light",
        7: "traffic sign",
        8: "train",
        9: "truck"
    }
}

with open(os.path.join(OUTPUT_DIR, "dataset.yaml"), "w") as f:
    yaml.dump(dataset_yaml, f, sort_keys=False)

print("dataset.yaml created successfully!")

In [ ]:
# READING THE dataset.yaml CONTENT

with open(os.path.join(OUTPUT_DIR, "dataset.yaml"), "r") as f:
    print(f.read())

In [ ]:
!pip install -q ultralytics

In [ ]:
import ultralytics
print(ultralytics.__version__)

In [ ]:
# LOADING THE MODEL

from ultralytics import YOLO

model = YOLO("yolo11m.pt")

In [ ]:
# TRAINING THE MODEL

results = model.train(
    data="/kaggle/input/datasets/mubeenakhund/dataset-yaml/dataset.yaml",

    # Training
    epochs=100,
    imgsz=640,
    batch=32,

    device=[0, 1],
    workers=8,
    amp=True,

    # Optimizer
    optimizer="AdamW",
    lr0=5e-4,
    weight_decay=5e-4,

    # Early stopping
    patience=10,

    # Saving
    project="YOLO11_BDD100K",
    name="yolo11m_detection_stage1",
    exist_ok=True,
    save=True,
    save_period=10,

    # Misc
    pretrained=True,
    verbose=True,
    plots=True
)

***THANK U***